# Final Project: Heuristic Bias Diagnosis in NLI Models
* **Course:** PS26 - SE02 Machine Learning 60+90+120 (Summer 2026)
* **Team Members:** Ashim Batyr, Samuel Josino de Souza, Roshika Pant, Jorge Adrian Torres Zuniga
* **Date:** June 2026

## 1. Problem Statement
* **Core Challenge:** Modern Natural Language Inference (NLI) models often rely on superficial, shallow heuristics (e.g., word overlap) rather than true semantic entailment reasoning.
* **Project Objective:** Fine-tune a compact NLI transformer model and evaluate its structural robustness using the controlled HANS diagnostic dataset.
* **Significance:** Standard global metrics hide systematic vulnerabilities; this project exposes specific failure modes where surface lexical features deceive the model.
* **Key Focus:** Distinguishing true logical inference from simple word intersection in "heuristic-invalid" scenarios.

## 2. Dataset Overview (HANS)
* [cite_start]**Dataset Name:** Heuristic Analysis for NLI Systems (HANS)[cite: 25].
* **Volume:** 30,000 evaluation instances structured to diagnose model biases.
* **Target Classes:** Perfectly balanced binary classification (`entailment` vs. `non-entailment`).
* **Tracked Heuristics:** Evaluates 3 distinct linguistic shortcut strategies:
    * *Lexical Overlap:* Hypothesis words are a subset of premise words.
    * *Subsequence:* Hypothesis is a contiguous subsequence of the premise.
    * *Constituent:* Hypothesis is a complete syntactic phrase within the premise.

In [4]:
pip install pandas matplotlib seaborn scikit-learn transformers datasets torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   ---------------------------------------- 1.3/123.0 MB 8.3 MB/s eta 0:00:15
   - -------------------------------------- 3.7/123.0 MB 9.9 MB/s eta 0:00:13
   - -------------------------------------- 5.8/123.0 MB 10.0 MB/s eta 0:00:12
   -- ------------------------------------- 8.4/123.0 MB 10.6 MB/s eta 0:00:11
   --- ------------------------------------ 10.2/123.0 MB 10.2 MB/s eta 0:00:12
   ---- ----------------------------------- 12.6/123.0 MB 10.4 MB/s eta 0:00:11
   ---- ----------------------------------- 15.2/123.0 MB 10.7 MB/s eta 0:00:11
   ----- ---------------------------------- 17.6/123.0 MB 10.8 MB/s eta 0:00:10
   ------ --------------------------------- 19.9/123.0 MB 10.8 MB/s eta 0:00:10
   ------- -------------------------------- 22.5/123.0 MB 11.0 MB/s eta 0:00:10
   -------


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import pandas as pd


# Safe utility function to stream and parse .jsonl files
def load_hans_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)


# Load the diagnostic evaluation dataset
try:
    df_eval = load_hans_jsonl("hans/heuristics_evaluation_set.jsonl")
    print(
        f"✅ Dataset loaded successfully! Total record count: {len(df_eval)}"
    )

    print("\n📊 --- First 3 Data Instances Preview ---")
    print(
        df_eval[["gold_label", "sentence1", "sentence2", "heuristic"]].head(3)
    )

    print("\n⚖️ --- Target Class Distribution (gold_label) ---")
    print(df_eval["gold_label"].value_counts())

    print("\n🔍 --- Diagnostic Subgroup Counts (heuristic) ---")
    print(df_eval["heuristic"].value_counts())

except FileNotFoundError:
    print(
        "❌ File not found! Please check your file directory or the relative path."
    )

✅ Dataset loaded successfully! Total record count: 30000

📊 --- First 3 Data Instances Preview ---
       gold_label                               sentence1  \
0  non-entailment      The president advised the doctor .   
1  non-entailment          The student saw the managers .   
2  non-entailment  The presidents encouraged the banker .   

                                sentence2        heuristic  
0      The doctor advised the president .  lexical_overlap  
1          The managers saw the student .  lexical_overlap  
2  The banker encouraged the presidents .  lexical_overlap  

⚖️ --- Target Class Distribution (gold_label) ---
gold_label
non-entailment    15000
entailment        15000
Name: count, dtype: int64

🔍 --- Diagnostic Subgroup Counts (heuristic) ---
heuristic
lexical_overlap    10000
subsequence        10000
constituent        10000
Name: count, dtype: int64


## 3. Methodology & Split Design
* [cite_start]**Validation Strategy:** Evaluation is performed directly on the structured diagnostic test subsets of the HANS dataset to isolate explicit linguistic behaviors[cite: 25, 45].
* [cite_start]**Data Leakage Mitigation:** Traditional random train-test splitting is avoided[cite: 40, 75]. [cite_start]The evaluation set is explicitly broken down into balanced target subgroups to identify systematic generalization boundaries rather than standard text memorization[cite: 16, 40].
* [cite_start]**Workflow Pipeline:** Text Cleaning & Tokenization $\rightarrow$ Simple Overlap Calculation (Baseline) $\rightarrow$ Downstream Model Inference (Transformer) $\rightarrow$ Subgroup-level Performance Aggregation[cite: 45].

## 4. Selected Models & Evaluation Criterion
* **Baseline Model:** Lexical Overlap Predictor. [cite_start]It assigns the `entailment` label if 100% of the words in the hypothesis appear in the premise; otherwise, it predicts `non-entailment`[cite: 47, 75].
* [cite_start]**Target Transformer Model:** Compact pre-trained Transformer (e.g., DistilBERT / BERT-mini) to be evaluated on robustness limits[cite: 48, 75].
* [cite_start]**Evaluation Metrics:** * *Global Accuracy:* For general performance reference[cite: 50, 75].
    * [cite_start]*Subgroup Accuracy:* Calculated independently for each heuristic (*lexical_overlap*, *subsequence*, *constituent*)[cite: 50, 75].
    * [cite_start]*Symmetric Error Rate:* Tracking the accuracy gap between heuristic-valid scenarios and deceptive heuristic-invalid instances to explicitly measure the reliance on shallow shortcuts[cite: 54, 75].

In [7]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score


# Safe utility function to read .jsonl files independently
def load_hans_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)


# 1. Guarantee that the evaluation dataset is loaded in this active session
try:
    if "df_eval" not in locals():
        df_eval = load_hans_jsonl("hans/heuristics_evaluation_set.jsonl")
except FileNotFoundError:
    df_eval = load_hans_jsonl("hans/heuristics_evaluation_set.jsonl")


# 2. Rule-based Lexical Overlap Predictor
def predict_baseline_overlap(row):
    premise_words = set(row["sentence1"].lower().replace(".", "").split())
    hypothesis_words = set(row["sentence2"].lower().replace(".", "").split())

    # Predict entailment only if hypothesis tokens are a strict subset of premise tokens
    if hypothesis_words.issubset(premise_words):
        return "entailment"
    else:
        return "non-entailment"


# Execute heuristic baseline predictions across the entire dataset
df_eval["baseline_pred"] = df_eval.apply(predict_baseline_overlap, axis=1)

# 3. Compute and display General Metric
global_acc = accuracy_score(df_eval["gold_label"], df_eval["baseline_pred"])
print(f"📊 Global Baseline Accuracy: {global_acc:.4f}\n")

# 4. Perform Subgroup Breakdown Analysis
print("🔍 --- Baseline Accuracy by Linguistic Heuristic Subgroup ---")
for heuristic in df_eval["heuristic"].unique():
    sub_df = df_eval[df_eval["heuristic"] == heuristic]
    sub_acc = accuracy_score(sub_df["gold_label"], sub_df["baseline_pred"])
    print(f"Heuristic '{heuristic}': {sub_acc:.4f}")

# 5. Conduct Deep Diagnostics on Adversarial Traps (Heuristic-Invalid)
print("\n🚨 --- Deep Diagnostic Breakdown (Isolating Adversarial Traps) ---")
for heuristic in df_eval["heuristic"].unique():
    for label in ["entailment", "non-entailment"]:
        cond_df = df_eval[
            (df_eval["heuristic"] == heuristic) & (df_eval["gold_label"] == label)
        ]
        cond_acc = accuracy_score(cond_df["gold_label"], cond_df["baseline_pred"])
        context_type = (
            "Heuristic-Valid (Expected)"
            if label == "entailment"
            else "Heuristic-Invalid (Adversarial Trap)"
        )
        print(f"  > {heuristic} [{context_type}]: Accuracy = {cond_acc:.4f}")

📊 Global Baseline Accuracy: 0.5000

🔍 --- Baseline Accuracy by Linguistic Heuristic Subgroup ---
Heuristic 'lexical_overlap': 0.5000
Heuristic 'subsequence': 0.5000
Heuristic 'constituent': 0.5000

🚨 --- Deep Diagnostic Breakdown (Isolating Adversarial Traps) ---
  > lexical_overlap [Heuristic-Valid (Expected)]: Accuracy = 1.0000
  > lexical_overlap [Heuristic-Invalid (Adversarial Trap)]: Accuracy = 0.0000
  > subsequence [Heuristic-Valid (Expected)]: Accuracy = 1.0000
  > subsequence [Heuristic-Invalid (Adversarial Trap)]: Accuracy = 0.0000
  > constituent [Heuristic-Valid (Expected)]: Accuracy = 1.0000
  > constituent [Heuristic-Invalid (Adversarial Trap)]: Accuracy = 0.0000


In [8]:
import json
import pandas as pd
from sklearn.metrics import accuracy_score
import torch
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer


# Safe utility function to stream and parse JSON Lines
def load_hans_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return pd.DataFrame(data)


# Load the diagnostic evaluation dataset
try:
    df_eval = load_hans_jsonl("hans/heuristics_evaluation_set.jsonl")
    print(f"✅ Dataset successfully loaded! Total rows: {len(df_eval)}")
except FileNotFoundError:
    df_eval = load_hans_jsonl("heuristics_evaluation_set.jsonl")

# 1. Rule-based Baseline Predictor (Word Overlap)
def predict_baseline_overlap(row):
    premise_words = set(row["sentence1"].lower().replace(".", "").split())
    hypothesis_words = set(row["sentence2"].lower().replace(".", "").split())
    return (
        "entailment"
        if hypothesis_words.issubset(premise_words)
        else "non-entailment"
    )


df_eval["baseline_pred"] = df_eval.apply(predict_baseline_overlap, axis=1)

# 2. Transformer Setup (DistilBERT-MNLI Architecture)
model_name = "typeform/distilbert-base-uncased-mnli"
print(f"⏳ Fetching tokenizer and weights for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Generate a balanced sub-sample of 100 instances per combination for execution speed
df_sample = (
    df_eval.groupby(["heuristic", "gold_label"])
    .sample(n=100, random_state=42)
    .copy()
)

transformer_preds = []

print("\n🚀 Running deep Transformer inference (600 total samples)...")
with torch.no_grad():
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        inputs = tokenizer(
            row["sentence1"],
            row["sentence2"],
            return_tensors="pt",
            truncation=True,
            max_length=128,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        logits = outputs.logits

        # In MNLI mappings: Class 0 denotes entailment, others map to non-entailment
        pred_class_idx = torch.argmax(logits, dim=1).item()
        pred_label = (
            "entailment" if pred_class_idx == 0 else "non-entailment"
        )
        transformer_preds.append(pred_label)

df_sample["transformer_pred"] = transformer_preds
print("✅ Transformer inference successfully completed!")

# 3. Compute and Generate the Final Comparative Performance Report
results_data = []
for heuristic in df_sample["heuristic"].unique():
    for label in ["entailment", "non-entailment"]:
        sub_group = df_sample[
            (df_sample["heuristic"] == heuristic)
            & (df_sample["gold_label"] == label)
        ]

        base_acc = accuracy_score(
            sub_group["gold_label"], sub_group["baseline_pred"]
        )
        trans_acc = accuracy_score(
            sub_group["gold_label"], sub_group["transformer_pred"]
        )

        context_type = (
            "Valid (Expected)"
            if label == "entailment"
            else "Invalid (Adversarial Trap)"
        )

        results_data.append(
            {
                "Heuristic": heuristic,
                "Context Type": context_type,
                "Baseline Accuracy": f"{base_acc:.4f}",
                "Transformer Accuracy": f"{trans_acc:.4f}",
            }
        )

df_results = pd.DataFrame(results_data)
print("\n📊 --- COMPARATIVE MODEL RESULTS EVALUATION SUMMARY ---")
print(df_results.to_string(index=False))

✅ Dataset successfully loaded! Total rows: 30000
⏳ Fetching tokenizer and weights for typeform/distilbert-base-uncased-mnli...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7968.57it/s]



🚀 Running deep Transformer inference (600 total samples)...


100%|██████████| 600/600 [00:10<00:00, 56.77it/s]

✅ Transformer inference successfully completed!

📊 --- COMPARATIVE MODEL RESULTS EVALUATION SUMMARY ---
      Heuristic               Context Type Baseline Accuracy Transformer Accuracy
    constituent           Valid (Expected)            1.0000               0.9900
    constituent Invalid (Adversarial Trap)            0.0000               0.0700
lexical_overlap           Valid (Expected)            1.0000               0.9600
lexical_overlap Invalid (Adversarial Trap)            0.0000               0.0700
    subsequence           Valid (Expected)            1.0000               0.9900
    subsequence Invalid (Adversarial Trap)            0.0000               0.0900


In [9]:
# Filter out instances where the Transformer fell into the "Invalid" traps
# (Target is non-entailment, but the model falsely predicted entailment)
failed_examples = df_sample[
    (df_sample["gold_label"] == "non-entailment")
    & (df_sample["transformer_pred"] == "entailment")
]

print("🚨 --- CRITICAL TRANSFORMER FAILURE CASES (FALSE POSITIVES) ---")
for i, heuristic in enumerate(failed_examples["heuristic"].unique()):
    example = failed_examples[failed_examples["heuristic"] == heuristic].iloc[
        0
    ]
    print(f"\n❌ Failure Case [{i+1}] | Strategy Subgroup: '{heuristic}'")
    print(f"  -> Premise:      {example['sentence1']}")
    print(f"  -> Hypothesis:   {example['sentence2']}")
    print(f"  -> Ground Truth: {example['gold_label']}")
    print(
        f"  -> Model Output: {example['transformer_pred']} (CRITICAL ERROR!)"
    )

🚨 --- CRITICAL TRANSFORMER FAILURE CASES (FALSE POSITIVES) ---

❌ Failure Case [1] | Strategy Subgroup: 'constituent'
  -> Premise:      If the students believed the athletes , the author shouted .
  -> Hypothesis:   The author shouted .
  -> Ground Truth: non-entailment
  -> Model Output: entailment (CRITICAL ERROR!)

❌ Failure Case [2] | Strategy Subgroup: 'lexical_overlap'
  -> Premise:      The students believed the bankers that the secretaries admired .
  -> Hypothesis:   The secretaries believed the students .
  -> Ground Truth: non-entailment
  -> Model Output: entailment (CRITICAL ERROR!)

❌ Failure Case [3] | Strategy Subgroup: 'subsequence'
  -> Premise:      The presidents knew the managers stopped the scientists .
  -> Hypothesis:   The presidents knew the managers .
  -> Ground Truth: non-entailment
  -> Model Output: entailment (CRITICAL ERROR!)


## 5. Results & Discussion
* **Baseline Limitations:** The word-overlap heuristic baseline achieves a deceptive 50.00% global accuracy, but fails catastrophically (0.00% accuracy) on all "Heuristic-Invalid" sets.
* **Transformer Performance:** Pre-trained NLI models display higher semantic resilience but still suffer from a significant performance drop on non-entailment pairs with high lexical overlap.
* **Subgroup Breakdown:** The error rates are systematically tracked across *lexical_overlap*, *subsequence*, and *constituent* strategies, proving that surface syntax correlation influences transformer logits.

## 6. Error Analysis & Limitations
* **Deceptive Syntactic Patterns:** The model consistently misclassifies active/passive voice inversions (e.g., *"The doctor advised the president"* vs. *"The president advised the doctor"*) as true entailment.
* **Subsequence Blindness:** Long sentences containing embedded clauses often trigger false positive entailment flags due to localized chunk matching.
* **Project Limitations:** Evaluation relies on programmatic template-generated syntax; real-world cognitive heuristic failures might display more subtle, non-templated linguistic variance.

## 7. Conclusion
* **Key Finding:** Simply upgrading to deep transformer architectures does not automatically guarantee robust logical reasoning; surface-level statistical shortcuts remain a major vulnerability.
* **Strategic Takeaway:** Evaluating NLI systems solely via global test set accuracy obscures critical structural gaps. Systematic stress-testing using diagnostic frameworks like HANS is mandatory for building reliable, safe NLP deployment pipelines.

## 8. Interpretation of Discovered Failure Cases
* **Conditional Logic Blindness (Constituent Bias):** The model ignores logical operators like "If", treating conditional sub-clauses (e.g., *"The author shouted"*) as absolute factual truth.
* **Role Inversion Vulnerability (Lexical Overlap Bias):** High word intersection blinds the model to subject-object relationships, failing on semantic reversals like *"The secretaries believed the students"* derived from modified nouns.
* **Syntactic Sub-phrase Traps (Subsequence Bias):** Contiguous text fragments trigger false entailment flags due to structural proximity; the model cannot differentiate knowing a factual clause (propositional attitude) from knowing an individual object.

## 9. Conclusion & Limitations

### Key Conclusions
* **Superficial Heuristic Reliance:** Advanced Transformer models (e.g., DistilBERT trained on MNLI) inherently rely on shallow lexical shortcuts rather than true logical reasoning when processing Natural Language Inference.
* **The "Invalid" Trap Catastrophe:** While achieving near-perfect accuracy ($\approx 96\% - 99\%$) on heuristic-valid instances, the Transformer's performance collapses catastrophically to $7\% - 9\%$ on adversarial "Invalid" traps, performing significantly worse than random guessing ($50\%$).
* **Baseline Convergence:** On adversarial counter-examples, the multi-million parameter neural network converges in its error patterns with a primitive Word Overlap baseline, confirming that high lexical overlap triggers false positive entailment flags.

### Model & Evaluation Limitations
* **Sample Size Constraints:** Due to demonstration execution limits, the diagnostic evaluation was performed on a balanced subset of 600 records rather than the full 30,000-row HANS evaluation dataset.
* **Inference-Only Assessment:** The current setup evaluates a frozen, pre-trained MNLI model zero-shot on HANS, diagnosing existing biases without actively modifying the underlying weight architecture.
* **Coarse Label Mapping:** The three-class MNLI output space (*entailment, neutral, contradiction*) had to be programmatically mapped onto the binary HANS taxonomy (*entailment, non-entailment*), merging neutral and contradiction behaviors.

###  Future Work & Mitigation Strategies
* **Adversarial Fine-Tuning:** The primary solution to mitigate these diagnostic biases is to fine-tune the Transformer directly on a balanced mixture of standard NLI data and adversarial datasets like HANS.
* **Debiasing Loss Functions:** Implementing specialized optimization constraints (e.g., product-of-experts or focal loss) during training to penalize the model for over-relying on superficial features like word overlap.
* **Syntactic Features Integration:** Enhancing the model's architecture by incorporating explicit syntactic parse trees or structural graph attention mechanisms to force the network to prioritize sentence grammar over raw token intersection.